# Week 02 — Notebooks, libraries and reproducible research
DE0823 · Digital Discourse Studies · Riga Technical University · 25 November 2026

This notebook accompanies the [interactive lesson](https://valrcs.github.io/Digital_Discourse_ETH713_Fall_2020/lessons/02-notebooks-libraries/). Core cells use only Python's standard library, no network access or pre-existing files. All texts are synthetic. Run all from a fresh kernel; optional installation and magic examples are Markdown, not automatically executed.

## 1. The environment
Python is the language; the kernel executes code. Colab, JupyterLab and VS Code provide interfaces. A notebook file stores cells, metadata and optional outputs, not live memory.

In [ ]:
import sys
from pathlib import Path
print(sys.version)
print(Path.cwd())

## 2. Code, Markdown and output
Markdown explains your question, method and limitations. Code executes instructions. The last expression may be displayed by IPython; print explicitly writes output.

In [ ]:
text = "Sveika, Rīga!\nDigital Humanities"
text

In [ ]:
print(text)

## 3. Runtime state
The visual order is not necessarily the execution order. The next three cells depend on each other. First run in order; then change the source and rerun only the final cell. Inspect old values. Restore the original source and restart/run all before proceeding.

In [ ]:
text = "Rīga Rīga Latvija"

In [ ]:
words = text.split()

In [ ]:
print(len(words))
assert len(words) == 3

### Intentional error example
Run this safely caught NameError. In a normal failed cell, read the last traceback line, then identify the missing name. Do not use broad exception handling to conceal errors.

In [ ]:
try:
    print(undefined_week02_name)
except NameError as error:
    print(type(error).__name__, error)

## 4. Modules, packages and libraries
Built-ins such as len require no import. Standard-library modules are distributed with Python; third-party distributions must be available in your environment. A package can contain submodules. A library is a broader collection of reusable functionality.

In [ ]:
import math
print(math.sqrt(81))

In [ ]:
import math as m
print(m.sqrt(81))

In [ ]:
from math import sqrt
from math import sqrt as square_root
print(sqrt(81), square_root(81))

Each import binds a name. Earlier imports remain in memory until removed or the kernel is restarted. In a clean namespace `import math as m` does not bind `math`.

In [ ]:
namespace = {}
exec('import math as m', namespace)
assert 'm' in namespace and 'math' not in namespace
print(namespace['m'].sqrt(81))

### Standard library: Counter
Counter counts hashable objects, including strings. It preserves insertion order for equal frequencies in most_common.

In [ ]:
from collections import Counter
counts = Counter(words)
print(counts.most_common())
assert counts['Rīga'] == 2

## 5. Installation and documentation
Installing makes a distribution available; importing binds a name. In a terminal use `python -m pip install requests`; in IPython use `%pip install requests`. The latter is a magic command, not Python syntax. The distribution `beautifulsoup4` exposes the import package `bs4`. These are optional commands; no installation is needed here. Only install trusted software and record versions.

In [ ]:
import importlib.util
print('requests available:', importlib.util.find_spec('requests') is not None)

In [ ]:
help(str.split)

In [ ]:
print([name for name in dir(str) if not name.startswith('_')][:12])

Read the signature, arguments, return values and examples. Consult [Python documentation](https://docs.python.org/3/library/) and [PyPI](https://pypi.org/). In Colab, a fresh runtime may have a different set of installed packages.

## 6. Files and UTF-8
Relative paths start at the current working directory. The notebook creates an isolated temporary directory so it will not overwrite existing research files. Temporary runtime files are not persistent; save important outputs separately.

In [ ]:
from tempfile import TemporaryDirectory
workspace = TemporaryDirectory(prefix='week02-')
root = Path(workspace.name)
data_dir = root / 'data'
results_dir = root / 'results'
data_dir.mkdir()
results_dir.mkdir()
print(root)

In [ ]:
documents = [
    'Rīga Jelgava Rīga',
    'Liepāja Rīga Cēsis',
    'Jelgava Liepāja Rīga',
]
source_path = data_dir / 'sample.txt'
source_path.write_text('\n'.join(documents) + '\n', encoding='utf-8');

In [ ]:
raw_text = source_path.read_text(encoding='utf-8')
print(raw_text)
assert raw_text.splitlines() == documents

In [ ]:
print(source_path.relative_to(root))
print(source_path.is_file())
print((data_dir / 'missing.txt').exists())

If a file is missing, verify its path and working directory. Do not assume that a file stored on GitHub is automatically present in Colab. The following error is caught only for demonstration.

In [ ]:
try:
    (data_dir / 'missing.txt').read_text(encoding='utf-8')
except FileNotFoundError as error:
    print(type(error).__name__)

## 7. Research question and provenance
How frequently do place-name tokens occur in these three synthetic Latvian documents? They are invented for teaching, not a historical corpus. Whitespace tokenization and optional lowercase normalization do not resolve inflection, punctuation or place-name ambiguity.

In [ ]:
normalize = True
analysis_documents = [line for line in raw_text.split('\n') if line.split()]
tokens = []
for document in analysis_documents:
    if normalize:
        document = document.lower()
    tokens.extend(document.split())
frequencies = Counter(tokens)
print(frequencies.most_common())
assert len(tokens) == 9
assert frequencies['rīga' if normalize else 'Rīga'] == 4

In [ ]:
print('Documents:', len(analysis_documents))
print('Tokens:', len(tokens))
print('Unique tokens:', len(frequencies))

In [ ]:
import csv
output_path = results_dir / 'counts.tsv'
with output_path.open('w', encoding='utf-8', newline='') as stream:
    writer = csv.writer(stream, delimiter='\t')
    writer.writerow(['token', 'count'])
    writer.writerows(frequencies.most_common())
print(output_path.read_text(encoding='utf-8'))

## 8. Reproducibility and limitations
Record input provenance, software and method parameters. Keep original input, preserve the procedure and restart/run all. A fresh execution is necessary but not sufficient to reproduce research in another environment. The default output contains nine tokens and four unique tokens. The count is not a count of all real-world references to Riga.

In [ ]:
import json
metadata = {'source': 'synthetic teaching corpus', 'python': sys.version.split()[0], 'normalization': 'lowercase' if normalize else 'none', 'tokenization': 'whitespace', 'documents': len(analysis_documents)}
metadata_path = results_dir / 'metadata.json'
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8');
print(metadata_path.read_text(encoding='utf-8'))

### Independent practice
1. Change a spelling or capitalization, regenerate the source file, and compare the counts. Restore the default example for the assertions, or deliberately update assertions to match your hypothesis.
2. Add a Markdown paragraph on data provenance and limitations.
3. Restart the kernel and run all. Export the notebook and required data/results to persistent storage before closing the runtime.
4. Explain why a saved output alone is not reproducible evidence.

## Optional bonus (not run automatically)
- `%pwd` shows the working directory; `%timeit len(words)` times an expression; `%%timeit` times a cell; `%run script.py` executes a script. These are IPython magics.
- `!ls` invokes a shell on compatible POSIX systems; Windows commands differ.
- For local environments, `python -m venv .venv` creates a virtual environment; select its kernel and document tested dependencies.
- A requirements file lists distributions and optionally tested versions. Do not blindly freeze an entire hosted environment.
- An `.ipynb` is JSON; Week 4 explores this format.
- Use GitHub for versioned source, never for credentials or private research data.

### Optional module demonstration (standard library, safe to run)
Write a small module in the isolated workspace and load it using an explicit path. In a normal project, a module on the import search path can be loaded with `import week02_helpers`. This explicit approach avoids changing the notebook working directory or overwriting an existing module.

In [ ]:
module_path = root / 'week02_helpers.py'
module_path.write_text('def count_words(text):\n    return len(text.split())\n', encoding='utf-8')
import importlib.util
spec = importlib.util.spec_from_file_location('week02_helpers', module_path)
helpers = importlib.util.module_from_spec(spec)
spec.loader.exec_module(helpers)
assert helpers.count_words('Rīga Latvija') == 2
print(helpers.count_words('Rīga Latvija'))

### Cleanup
The temporary directory exists for the lifetime of the workspace object. This optional cleanup code removes only the generated teaching files. Save anything you need first. Core execution leaves files available for inspection.

In [ ]:
print('Teaching files:', [str(p.relative_to(root)) for p in sorted(root.rglob('*')) if p.is_file()])
# Optional after inspection: workspace.cleanup()